# 06 — Training

**What this notebook does.** Trains the boundary U-Net on one fold and
measures it honestly: IoU, Dice, Precision, Recall and a 2-px-tolerance
boundary F-score, reported **per dataset within the validation split** as well
as pooled, with the three loss terms logged separately and a per-epoch
diagnostic that says whether the clDice term is measuring anything at all on
real predictions. Pixel accuracy is never reported.

**What must already exist.**

- `GH_TOKEN` as a host secret and the datasets mounted — `00_bootstrap.ipynb`
- `reports/manifests/<fold>.csv` and `configs/fold_stats.yaml` — `03_tiling.ipynb`
- `configs/dataloader.yaml` with an entry for **this** host — `04_dataset.ipynb`
- `configs/default.yaml` with `model:`, `loss:` and `train.batch_size` — `05_model_and_loss.ipynb`

**What it produces.** `PERSISTENT_DIR/checkpoints/<fold>/{last,best}.pt`,
TensorBoard logs under `PERSISTENT_DIR/logs/<fold>/`, and
`reports/train_<fold>.{json,md}` pushed back to the repo. No config value is
written by this notebook: step 6's hyperparameters were decided on `main` and
are read, not measured here.

**Expected runtime on a free T4.** ~2.5 min/epoch for `dev` (2,333 train tiles
at batch 64), so ~100 minutes for the full 40 epochs — longer than a free
session is guaranteed to last, which is what the next cell is about.

# If your session dies

**It will.** Colab and Kaggle free tiers kill sessions without warning, and a
40-epoch run is longer than either guarantees. Nothing about that is a
disaster here, and the recovery is not a special procedure:

> **Re-run this notebook from the top.** That is the whole recovery.

Why it works:

1. **Every epoch is checkpointed.** `src/train.py` writes `last.pt` after every
   single epoch, atomically — to a temporary file, `fsync`, then rename — so a
   session killed mid-write leaves the *previous* epoch's checkpoint intact
   rather than a truncated file that fails to load hours later. You lose at
   most the epoch that was running.
2. **The resume cell below is automatic.** It finds `last.pt`, verifies that
   its fold name and config hash match this run, and continues from the next
   epoch. It does not ask. If either check fails it *refuses* rather than
   silently continuing a curve whose two halves are different experiments.
3. **The tile cache comes back warm.** The decoded source images are one file
   under `PERSISTENT_DIR`, so the loaders are ready in seconds instead of the
   ~27 minutes a cold decode off Drive costs. Cell 7 prints `WARM` or `COLD`;
   if it says COLD on a resume, something moved and it is worth stopping to
   find out what before spending the time.

So a resume costs about a minute. What it does **not** survive:

- **Kaggle's `/kaggle/working` is wiped when the session ends.** On Kaggle the
  checkpoints are already gone by the time you come back. Save the output as a
  dataset version, or run this on Colab where `PERSISTENT_DIR` is Drive. The
  last cell says this again, loudly, on the host where it applies.
- Changing a hyperparameter between sessions. That changes the config hash and
  the resume will refuse — deliberately. Start a clean run instead.

## Cell 1 — the standard bootstrap block

Identical in every notebook. Reads `GH_TOKEN` from the host secret store,
fetches `scripts/bootstrap_session.py`, then hands over to `bootstrap()`,
which syncs the repo, installs what is missing, mounts Drive on Colab and
returns `PATHS`.

In [ ]:
# --- standard bootstrap block: identical in every notebook ---------------
OWNER, REPO, BRANCH = "arhorri", "boundary", "main"

import importlib, os, pathlib, sys, urllib.request


def _gh_token():
    """Read GH_TOKEN from whichever secret store this host provides."""
    try:
        from google.colab import userdata

        return userdata.get("GH_TOKEN")
    except Exception:
        pass
    try:
        from kaggle_secrets import UserSecretsClient

        return UserSecretsClient().get_secret("GH_TOKEN")
    except Exception:
        pass
    return os.environ.get("GH_TOKEN")


_token = _gh_token()
if not _token:
    raise SystemExit(
        "GH_TOKEN secret is missing.\n"
        "  Colab : key icon in the left sidebar -> add GH_TOKEN -> notebook access ON\n"
        "  Kaggle: Add-ons -> Secrets -> add GH_TOKEN -> attach to this notebook"
    )

_req = urllib.request.Request(
    f"https://api.github.com/repos/{OWNER}/{REPO}/contents/scripts/bootstrap_session.py?ref={BRANCH}",
    headers={
        "Authorization": f"Bearer {_token}",
        "Accept": "application/vnd.github.raw",
    },
)
pathlib.Path("bootstrap_session.py").write_bytes(urllib.request.urlopen(_req).read())
del _token

if str(pathlib.Path.cwd()) not in sys.path:
    sys.path.insert(0, str(pathlib.Path.cwd()))
import bootstrap_session

bootstrap_session = importlib.reload(bootstrap_session)

PATHS = bootstrap_session.bootstrap(
    repo_url=f"https://github.com/{OWNER}/{REPO}.git", branch=BRANCH
)

## Choose the fold, and read the hyperparameters that follow from it

`FOLD` is the only thing to change in this notebook. Everything else is
resolved from it: which manifest is loaded, which dataset is held out, which
`pos_weight` the loss carries, which sampler weights the loader uses, and where
the checkpoints go.

`dev` is the alias for `fold_uhcs2` — same protocol, smallest validation set,
fastest to iterate on. The real folds are `fold_MetalDam`, `fold_uhcs1` and
`fold_uhcs2`; `test` (Steel2) is never trained or validated on.

**Nothing below is a literal.** The cell prints the source of every value
alongside it, because the whole point of steps 3–5 was that these get measured
once and read afterwards:

| value | comes from |
| --- | --- |
| `pos_weight` | `configs/fold_stats.yaml`, this fold's train split |
| sampler weights | `configs/fold_stats.yaml`, by parent |
| `batch_size` | `configs/default.yaml`, measured in notebook 05 |
| `num_workers` | `configs/dataloader.yaml`, **this host's** entry from notebook 04 |
| everything else | `configs/default.yaml` `train:` / `model:` / `loss:` |

### What a wrong `pos_weight` looks like in the metrics

It does not raise, and the loss curve looks fine either way. What changes is
the shape of the failure, and it is legible in Precision and Recall — which is
one of the reasons they are never collapsed into F1 alone here:

- **Too low** (or absent): recall collapses towards zero while precision looks
  respectable or even excellent. The model has discovered that predicting
  almost no boundary is a good deal — background is 85–95% of pixels — and the
  few boundary pixels it does emit are the easy, obvious ones. A recall under
  ~0.2 after a few epochs, with the predicted boundary fraction far below the
  true one, is this failure and not a hard fold.
- **Too high**: the opposite. Recall runs ahead of precision, the predicted
  boundary fraction overshoots the true one, and the output is thick smeared
  bands that touch every real boundary and a lot of grain interior too.

The per-epoch table prints `true_fraction` and `pred_fraction` next to the
metrics for exactly this reason: if the model is emitting 1% boundary where the
truth is 5%, no F1 number will tell you that as directly.

In [ ]:
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

from src import dataset as ds
from src import losses as losses_mod
from src import model as model_mod
from src import train as train_mod

FOLD = "dev"          # dev | fold_MetalDam | fold_uhcs1 | fold_uhcs2
EPOCHS = None         # None -> train.epochs from the config
FIGURE_EVERY = 1      # epochs between the per-dataset prediction panels


def progress(seq, desc=""):
    return tqdm(seq, desc=desc, leave=False)


train_settings = train_mod.load_config()
model_settings = model_mod.load_config()
loss_settings = losses_mod.load_config()
ds_settings = ds.load_config()
fold_stats = ds.load_fold_stats()
entry = fold_stats["folds"][FOLD]

trainer = train_mod.Trainer(fold=FOLD, resolved=PATHS, settings=train_settings,
                            model_settings=model_settings,
                            loss_settings=loss_settings,
                            dataset_settings=ds_settings)

print(f"fold {FOLD}  (alias of {entry.get('alias_of') or '-'}, "
      f"held out: {entry['held_out']})")
print(f"  train {entry['n_train_tiles']} tiles / {entry['n_train_parents']} parents "
      f"from {entry['train_datasets']}")
print(f"  val   {entry['n_val_tiles']} tiles / {entry['n_val_parents']} parents "
      f"from {entry['val_datasets']}")
print(f"  boundary fraction: train mean {entry['train_boundary_fraction']['mean']:.4f}, "
      f"val mean {entry['val_boundary_fraction']['mean']:.4f} "
      f"({entry['val_boundary_fraction']['mean'] / entry['train_boundary_fraction']['mean']:.2f}x)")

rows = [
    ("pos_weight", losses_mod.fold_pos_weight(FOLD, fold_stats=fold_stats),
     f"configs/fold_stats.yaml folds.{FOLD}.pos_weight"),
    ("batch_size", train_settings["batch_size"],
     "configs/default.yaml train.batch_size (measured, notebook 05)"),
    ("num_workers", trainer.resolve_num_workers(),
     trainer.sources["num_workers"]),
    ("epochs", EPOCHS or train_settings["epochs"], "configs/default.yaml train.epochs"),
    ("lr", train_settings["lr"], "configs/default.yaml train.lr"),
    ("encoder lr", float(train_settings["lr"]) * float(train_settings["encoder_lr_scale"]),
     "train.lr * train.encoder_lr_scale"),
    ("weight_decay", train_settings["weight_decay"], "configs/default.yaml"),
    ("warmup_epochs", train_settings["warmup_epochs"], "configs/default.yaml"),
    ("grad_clip", train_settings["grad_clip"], "configs/default.yaml"),
    ("amp", train_settings["amp"], "configs/default.yaml train.amp"),
    ("seed", train_settings["seed"], "configs/default.yaml train.seed"),
    ("encoder", model_settings["encoder"], "configs/default.yaml model.encoder"),
    ("freeze_encoder_epochs", model_settings["freeze_encoder_epochs"],
     "configs/default.yaml model.freeze_encoder_epochs"),
    ("w_bce / w_dice / w_cldice",
     f"{loss_settings['w_bce']} / {loss_settings['w_dice']} / {loss_settings['w_cldice']}",
     "configs/default.yaml loss:"),
    ("patch_size", ds_settings["patch_size"], "configs/default.yaml dataset.patch_size"),
    ("config hash", trainer.hash, "sha256 of model+loss+train+dataset settings"),
]
print()
print(f"  {'value':<26}{'resolved':<22}source")
for name, value, source in rows:
    print(f"  {name:<26}{str(value):<22}{source}")

others = {name: fold_stats["folds"][name]["pos_weight"]
          for name in ("fold_MetalDam", "fold_uhcs1", "fold_uhcs2")}
print(f"\npos_weight across folds: {others} -- this fold's value is not "
      "transferable and its loss values are not comparable to the others'.")

## Build the loaders, and confirm the cache and the alignment BEFORE spending GPU hours

Two things get confirmed here, and both are cheap now and expensive later.

**Did the tile cache come back warm?** A cold build decodes ~1,000 source
images off Drive at roughly 0.6 s each — about 27 minutes, paid again on every
session start and every crash resume. `WARM` means one sequential read of the
file step 4 wrote. On a resume this should always say WARM; if it says COLD,
the manifest or the crops changed and the cache key no longer matches, which is
worth understanding *before* training on top of it.

**Are the image and the mask still aligned?** The augmentation pipeline applies
spatial transforms to both as one `Compose` and photometric transforms to the
image only. Step 4 tested that. It is still worth looking at four augmented
pairs here, because a broken alignment produces a model that trains, converges,
and is worthless — and you would not find out for two hours.

In [ ]:
summary = trainer.setup(progress=progress)

print(f"device {summary['device']}  "
      f"{summary['gpu'] or ''}  AMP {summary['amp']}")
print(f"seed {summary['seed']['seed']}: {summary['seed']['note']}")
print(f"steps/epoch {summary['steps_per_epoch']}  "
      f"batch {summary['batch_size']}  workers {summary['num_workers']}")
print(f"parameters {summary['parameters']['total']:,} total, "
      f"encoder {summary['parameters']['encoder_total']:,}")
print(f"checkpoints -> {summary['checkpoint_dir']}")
print(f"tensorboard -> {summary['log_dir']}")

for split, info in summary["cache"].items():
    flag = "WARM" if info["source"] == "warm" else "COLD"
    print(f"\ntile cache {split}: {flag} -- {info['images']} images in "
          f"{info['seconds']:.1f}s")
    if flag == "COLD":
        print("  ! a cold build means the cache key changed (manifest or crops). "
              "That is fine on a first run and suspicious on a resume.")

composition = pd.DataFrame([
    {"split": split, "dataset": name, "tiles": info["tiles"],
     "parents": info["parents"]}
    for split, comp in (("train", summary["train_composition"]),
                        ("val", summary["val_composition"]))
    for name, info in comp.items()])
print("\nsplit composition (the val rows are why nothing is reported pooled only):")
print(composition.to_string(index=False))
print(f"\nheld-out dataset for this fold: {summary['held_out']} -- "
      f"best checkpoints are selected on ITS Dice, not the pooled number.")

## Four augmented training pairs

The mask is drawn over the image in red. If the spatial transform were applied
to only one of the two, the red lines would sit beside the features they are
supposed to trace rather than on them. This is the cheapest possible check and
it is the last chance to catch it before the GPU time starts.

In [ ]:
import matplotlib.pyplot as plt

picks = np.random.default_rng(0).choice(len(trainer.train_ds), size=4, replace=False)
fig, axes = plt.subplots(1, 4, figsize=(18, 4.8))
for ax, index in zip(axes, picks):
    sample = trainer.train_ds[int(index)]
    image, mask = sample["image"][0], sample["mask"][0]
    shown = (image - image.min()) / max(1e-6, float(np.ptp(image)))
    rgb = np.dstack([shown] * 3)
    rgb[..., 0] = np.where(mask > 0, 1.0, rgb[..., 0])
    rgb[..., 1] = np.where(mask > 0, rgb[..., 1] * 0.2, rgb[..., 1])
    rgb[..., 2] = np.where(mask > 0, rgb[..., 2] * 0.2, rgb[..., 2])
    ax.imshow(rgb)
    ax.set_title(f"{sample['dataset']}\n{sample['tile_id']}", fontsize=8)
    ax.axis("off")
plt.tight_layout()
plt.show()

## Auto-resume

This is the cell that makes a killed session a non-event. It looks for
`last.pt` under `PERSISTENT_DIR/checkpoints/<fold>/` and, if it finds one:

- **checks the fold name.** Resuming a `fold_uhcs1` checkpoint into a
  `fold_MetalDam` run would train with `pos_weight` 15.974 on weights fitted
  under 7.111 and validate against the wrong held-out dataset. Nothing about
  that raises on its own, so it is checked here and refused.
- **checks the config hash** — a hash of the model, loss, train and dataset
  settings. A resume that quietly adopts a new learning rate produces a curve
  whose first half and second half are different experiments. Refused, with a
  printed diff of exactly which keys changed.

Both refusals are exceptions, not warnings. If you meant to change something,
start a clean run; if you have decided the difference is immaterial, pass
`allow_config_change=True` and say so in the report.

In [ ]:
status = trainer.maybe_resume()

if status["resumed"]:
    print(f"RESUMING from {status['path']}")
    print(f"  {status['reason']}")
    print(f"  {status['epochs_done']} epoch(s) already done; continuing at "
          f"epoch {status['start_epoch']}")
    print(f"  RNG state restored: {status['rng_restored']}")
    print(f"  best so far: {status['best']}")
    print(f"  saved {status['saved_utc']} with AMP={status['saved_amp']}")
    if len(trainer.history) >= 2:
        print(f"  train loss so far: {trainer.history[0]['train']['total']:.4f} "
              f"-> {trainer.history[-1]['train']['total']:.4f}")
else:
    print(f"STARTING CLEAN -- {status['reason']}")
    print(f"  checkpoints will be written to {trainer.checkpoint_dir}")
print(f"\nconfig hash for this run: {trainer.hash}")

## TensorBoard

Everything logged below is also written to TensorBoard: the three loss terms
separately, every metric pooled and per dataset, the learning rate, the
trainable parameter count, and the clDice probe. The `logdir` is the whole
`logs/` tree, so folds trained in different sessions appear side by side.

If it does not render (Kaggle sometimes blocks the proxy), the printed tables
below carry the same numbers and the log files survive for later.

In [ ]:
logdir = Path(PATHS["persistent_dir"]) / train_settings["log_subdir"]
logdir.mkdir(parents=True, exist_ok=True)
print(f"tensorboard logdir: {logdir}")
try:
    ip = get_ipython()
    ip.run_line_magic("load_ext", "tensorboard")
    ip.run_line_magic("tensorboard", f"--logdir {logdir}")
except Exception as exc:
    print(f"inline TensorBoard unavailable ({type(exc).__name__}: {exc}).")
    print("The logs are still being written; the tables below carry the same "
          "numbers.")

## Train

One block of output per epoch, in this order:

1. **the three loss terms separately**, train and val. BCE, Dice and clDice
   differ by orders of magnitude — on an empty prediction BCE is ~25 and Dice
   is ~1 — so a falling total says almost nothing about which term is moving.
2. **the metrics per dataset**, then pooled. The held-out dataset's row is
   marked; it is the measurement. The pooled row is a footnote and is labelled
   as one. `true_frac` and `pred_frac` sit next to them because the fastest way
   to read a `pos_weight` problem is the gap between those two numbers.
3. **the clDice probe** — see the section after this one.
4. **three panels per validation dataset**: raw tile, ground truth,
   prediction. One row per dataset, so the held-out set is not averaged away
   visually either. The tile is the one at the median boundary fraction for its
   dataset, fixed for the whole run, so what changes across epochs is the
   model and not the sample.

The checkpoint is written after every epoch before any of this prints, so
interrupting here is safe.

In [ ]:
def show_predictions(record, trainer):
    examples = trainer.example_predictions(per_dataset=1)
    fig, axes = plt.subplots(len(examples), 3,
                             figsize=(13, 4.4 * len(examples)))
    axes = np.atleast_2d(axes)
    for row, example in zip(axes, examples):
        held = " (HELD OUT)" if example["dataset"] == trainer.held_out else ""
        row[0].imshow(example["image"], cmap="gray")
        row[0].set_title(f"{example['dataset']}{held}\n{example['tile_id']}",
                         fontsize=9)
        row[1].imshow(example["truth"], cmap="magma", vmin=0, vmax=1)
        row[1].set_title(f"ground truth  ({example['truth'].mean():.3f})",
                         fontsize=9)
        row[2].imshow(example["prob"], cmap="magma", vmin=0, vmax=1)
        row[2].set_title(f"prediction, epoch {record['epoch']}  "
                         f"(>= {trainer.settings['threshold']}: "
                         f"{example['pred'].mean():.3f})", fontsize=9)
        for ax in row:
            ax.axis("off")
    plt.tight_layout()
    plt.show()


def on_epoch_end(record, trainer):
    eta = record["eta_seconds"]
    print(f"\n{'=' * 100}")
    print(f"epoch {record['epoch']:>3}/{(EPOCHS or trainer.settings['epochs']) - 1}"
          f"   {record['seconds']:.1f}s   ETA {eta / 60:.1f} min"
          f"   lr {record['lr']:.3e}"
          f"   trainable {record['trainable_params']:,}"
          f"{'  [encoder FROZEN]' if record['encoder_frozen'] else ''}"
          f"{'   <<< BEST so far' if record['is_best'] else ''}")
    if record["freeze_changed"] and not record["encoder_frozen"]:
        transition = trainer.freeze_transitions[-1]
        print(f"  ENCODER UNFROZEN: trainable "
              f"{transition['trainable_before']:,} -> "
              f"{transition['trainable_after']:,}; optimizer now holds "
              f"{transition['optimized_after']:,} parameters in "
              f"{transition['groups']} groups")

    loss_table = pd.DataFrame([
        {"split": split, **{k: record[split][k]
                            for k in ("total", "bce", "dice", "cldice")}}
        for split in ("train", "val")])
    print("\n  loss terms (they differ by orders of magnitude -- read them apart):")
    print("   " + loss_table.to_string(index=False,
                                       float_format=lambda v: f"{v:9.4f}").replace("\n", "\n   "))

    metric_rows = []
    for name, metrics in record["metrics"]["per_dataset"].items():
        metric_rows.append({
            "dataset": name + (" <-HELD OUT" if name == trainer.held_out else ""),
            **{k: metrics[k] for k in train_mod.METRIC_ORDER},
            "true_frac": metrics["true_fraction"],
            "pred_frac": metrics["pred_fraction"],
            "tiles": metrics["tiles"]})
    pooled = record["metrics"]["pooled"]
    metric_rows.append({"dataset": "pooled (footnote)",
                        **{k: pooled[k] for k in train_mod.METRIC_ORDER},
                        "true_frac": pooled["true_fraction"],
                        "pred_frac": pooled["pred_fraction"],
                        "tiles": pooled["tiles"]})
    print("\n  validation metrics -- per dataset is the headline, pooled is the footnote:")
    print("   " + pd.DataFrame(metric_rows).to_string(
        index=False, float_format=lambda v: f"{v:7.4f}").replace("\n", "\n   "))

    probe = record["cldice_probe"]
    print(f"\n  clDice probe on {probe['tiles']} fixed val tiles: "
          f"skeleton delta {probe['skeleton_delta']:.6f} "
          f"(max {probe['skeleton_delta_max']:.4f})  ->  "
          f"{'DEGENERATE' if probe['degenerate'] else 'ACTIVE'}")
    print(f"    skel(pred) {probe['skel_pred_sum']:.0f} px, "
          f"skel(true) {probe['skel_true_sum']:.0f} px, "
          f"skel(pred) on true {probe['skel_pred_on_true']:.0f} px, "
          f"t_prec {probe['t_prec']:.4f}, t_rec {probe['t_rec']:.4f}")
    print(f"    Dice term {probe['dice']:.4f}   clDice term {probe['cldice']:.4f}")

    if record["epoch"] % FIGURE_EVERY == 0:
        show_predictions(record, trainer)


started = time.perf_counter()
history = trainer.fit(epochs=EPOCHS, on_epoch_end=on_epoch_end, progress=progress)
print(f"\ntraining finished: {len(history)} epochs in "
      f"{(time.perf_counter() - started) / 60:.1f} min")
print(f"best epoch {trainer.best['epoch']} by Dice on {trainer.best['key']}: "
      f"{trainer.best['metric']:.4f}")

## The whole run in three tables

Scrollback is lost when a session dies; the history lives in the checkpoint, so
this cell rebuilds every table from `trainer.history` and works identically
after a resume.

The clDice table is the one to read carefully. `skeleton_delta` is
`mean |soft_skeleton(sigmoid(logits)) - sigmoid(logits)|` over a fixed sample
of validation tiles. The soft skeleton returns its input unchanged wherever
erosion cannot bite, so a delta pinned near zero for every epoch means clDice
is computing Dice with a slightly different denominator and contributing
nothing. That is a finding, and it goes in the report.

In [ ]:
loss_history = pd.DataFrame([
    {"epoch": r["epoch"],
     **{f"train_{k}": r["train"][k] for k in ("total", "bce", "dice", "cldice")},
     **{f"val_{k}": r["val"][k] for k in ("total", "bce", "dice", "cldice")},
     "lr": r["lr"], "trainable": r["trainable_params"],
     "seconds": r["seconds"]}
    for r in trainer.history])
print("loss terms per epoch")
print(loss_history.to_string(index=False, float_format=lambda v: f"{v:9.4f}"))

metric_history = pd.DataFrame([
    {"epoch": r["epoch"], "dataset": name,
     "held_out": name == trainer.held_out,
     **{k: metrics[k] for k in train_mod.METRIC_ORDER},
     "true_frac": metrics["true_fraction"], "pred_frac": metrics["pred_fraction"]}
    for r in trainer.history
    for name, metrics in list(r["metrics"]["per_dataset"].items())
    + [("pooled (footnote)", r["metrics"]["pooled"])]])
print("\nvalidation metrics per epoch, per dataset")
print(metric_history.to_string(index=False, float_format=lambda v: f"{v:7.4f}"))

probe_history = pd.DataFrame([
    {"epoch": r["epoch"],
     **{k: r["cldice_probe"][k] for k in
        ("skel_pred_sum", "skel_true_sum", "skel_pred_on_true", "t_prec",
         "t_rec", "skeleton_delta", "dice", "cldice")},
     "degenerate": r["cldice_probe"]["degenerate"]}
    for r in trainer.history])
print("\nclDice diagnostic on real predictions")
print(probe_history.to_string(index=False, float_format=lambda v: f"{v:10.6f}"))
degenerate = int(probe_history["degenerate"].sum())
print(f"\n{degenerate} of {len(probe_history)} epochs came back DEGENERATE. "
      f"{trainer.history[-1]['cldice_probe']['verdict']}")

fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))
for term in ("total", "bce", "dice", "cldice"):
    axes[0].plot(loss_history["epoch"], loss_history[f"val_{term}"], label=term)
axes[0].set_yscale("log"); axes[0].set_title("validation loss terms (log)")
axes[0].set_xlabel("epoch"); axes[0].legend(); axes[0].grid(alpha=0.3)

for name, group in metric_history.groupby("dataset"):
    style = "-" if group["held_out"].any() else "--"
    axes[1].plot(group["epoch"], group["dice"], style, label=name)
axes[1].set_title("Dice per dataset (solid = held out)")
axes[1].set_xlabel("epoch"); axes[1].legend(fontsize=8); axes[1].grid(alpha=0.3)

held = metric_history[metric_history["held_out"]]
axes[2].plot(held["epoch"], held["precision"], label="precision")
axes[2].plot(held["epoch"], held["recall"], label="recall")
axes[2].plot(held["epoch"], held["boundary_f"], label="boundary-F (2 px)")
axes[2].set_title(f"{trainer.held_out}: precision vs recall, separately")
axes[2].set_xlabel("epoch"); axes[2].legend(); axes[2].grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Reading these numbers

**Read Precision and Recall separately. Never F1 alone.** They fail in
opposite directions and the fix differs. There is also an asymmetry specific to
what this model is *for*: it is a prior for a downstream watershed, and in that
role **precision matters more than recall**. A false boundary splits a region
permanently — the watershed obeys it and there is nothing downstream that can
undo it. A missed boundary leaves two regions merged, which the unsupervised
stage still has a chance to separate on intensity or texture. Prefer the
checkpoint that is a little conservative over the one that is a little
enthusiastic, and if you tune the threshold in step 7, tune it upward.

**Recall near zero means `pos_weight` is too low.** Not that the fold is hard.
The model has found the local optimum of predicting almost no boundary, which
scores 85–95% of pixels correct — which is why pixel accuracy appears nowhere
in this notebook. The tell is `pred_frac` far below `true_frac` in the metric
table. Raising `pos_weight` is not a free parameter here though: it is measured
from the fold's own train split, so if it looks wrong the thing to check is
whether the right fold's value is being loaded, not whether a different number
would train better.

**Rising validation loss with rising validation Dice is normal here, not
overfitting.** The loss is dominated by weighted BCE, which is a per-pixel
*calibration* measure: as the model gets more confident, its mistakes get more
expensive faster than its correct pixels get cheaper. Dice and the boundary
F-score measure *overlap*, which is what actually matters downstream. Trust the
overlap metrics on the held-out dataset. This is also why `best.pt` is selected
on held-out Dice and not on validation loss.

**If the boundaries come out BROKEN, `w_cldice` is not the lever.** Step 5
measured what clDice does, and it is not what its name suggests on this data:
it is nearly blind to line *thickness* (dilated: Dice 0.335/0.390 versus clDice
0.007/0.000), which is why it is in the loss — it stops Dice from spending the
gradient on a width that step 2 chose by convention. It does **not** add
sensitivity to gaps. Turning `w_cldice` up buys more thickness tolerance and
approximately nothing else. The levers that would actually change broken
boundaries are:

1. a thicker `boundary_gt.line_width_px`, which would give the soft skeleton a
   real centreline to compare — this means re-running **steps 2 and 3**, not
   editing a weight; or
2. an explicit connectivity metric at evaluation time (component count,
   or the watershed's region count against the truth's), so the failure is at
   least measured rather than hoped about.

Whichever is chosen, write it down here. Do not silently retune the weights —
that is how a project ends up with a loss nobody can explain.

**`fold_MetalDam`'s numbers are not comparable to the other folds'.** It trains
at boundary fraction 0.059 and validates at 0.154 — a 2.6× density mismatch —
because leave-one-dataset-out holds out the densest set. That is why it carries
`pos_weight` 15.974 against roughly 7.1 for the others. A BCE term scaled by
15.974 and one scaled by 7.111 are different units: comparing the two folds'
loss curves is meaningless. Compare them on Dice, boundary-F and precision/
recall on their held-out datasets, and expect `fold_MetalDam` to score lower
without that being a bug.

## Checks

Where the step is declared correct or not.

In [ ]:
checks = []


def check(name, ok, detail=""):
    checks.append((name, bool(ok)))
    print(f"{'PASS' if ok else 'FAIL'}  {name}{'  -- ' + detail if detail else ''}")


history = trainer.history
check("more than one epoch completed", len(history) >= 2,
      f"{len(history)} epochs in trainer.history")
if len(history) >= 2:
    first, last = history[0]["train"]["total"], history[-1]["train"]["total"]
    check("training loss decreased from epoch 1", last < first,
          f"{first:.4f} -> {last:.4f} over {len(history)} epochs")
    first_val, last_val = history[0]["val"]["total"], history[-1]["val"]["total"]
    print(f"      (validation loss {first_val:.4f} -> {last_val:.4f}; rising "
          "here is normal -- see 'Reading these numbers')")

# -- checkpoints ----------------------------------------------------------
persistent = Path(PATHS["persistent_dir"]).resolve()
check("checkpoint exists under PERSISTENT_DIR",
      trainer.last_path.is_file()
      and persistent in trainer.last_path.resolve().parents,
      f"{trainer.last_path} "
      f"({trainer.last_path.stat().st_size / 1024 ** 2:.0f} MB)"
      if trainer.last_path.is_file() else "missing")
check("best checkpoint written", trainer.best_path.is_file(),
      f"{trainer.best_path} "
      f"({trainer.best_path.stat().st_size / 1024 ** 2:.0f} MB), epoch "
      f"{trainer.best['epoch']}, {trainer.best['key']} Dice "
      f"{trainer.best['metric']:.4f}" if trainer.best_path.is_file() else "missing")

state = train_mod.load_checkpoint(trainer.last_path)
required = {"model", "optimizer", "scheduler", "scaler", "rng", "epoch",
            "fold", "config_hash", "history"}
check("checkpoint reloads and carries everything a resume needs",
      required <= set(state),
      f"missing {sorted(required - set(state))}" if not required <= set(state)
      else f"epoch {state['epoch']}, {len(state['history'])} history records")
reload_ok = True
try:
    probe_model = model_mod.build_model(settings=model_settings)
    probe_model.load_state_dict(state["model"])
except Exception as exc:
    reload_ok = False
    reload_detail = f"{type(exc).__name__}: {exc}"
else:
    reload_detail = "state_dict loads into a freshly built model"
check("checkpoint weights load into a fresh model", reload_ok, reload_detail)

# -- the resume guards actually refuse ------------------------------------
guard_dir = Path(PATHS["persistent_dir"]) / "checkpoints" / "_guard_probe"
guard_dir.mkdir(parents=True, exist_ok=True)
wrong_fold = guard_dir / "wrong_fold.pt"
wrong_hash = guard_dir / "wrong_hash.pt"
train_mod.atomic_save({"fold": "fold_not_this_one", "epoch": 0,
                       "config_hash": trainer.hash, "hashed_config": {}},
                      wrong_fold)
train_mod.atomic_save({"fold": trainer.fold, "epoch": 0,
                       "config_hash": "0000deadbeef0000",
                       "hashed_config": {}}, wrong_hash)
refused = {}
for label, path in (("fold", wrong_fold), ("config hash", wrong_hash)):
    try:
        trainer.maybe_resume(path=path)
        refused[label] = False
    except train_mod.TrainError:
        refused[label] = True
check("resume REFUSES a checkpoint from another fold", refused["fold"],
      "a fold_uhcs1 checkpoint in a fold_MetalDam run would train under the "
      "wrong pos_weight")
check("resume REFUSES a checkpoint under a different config hash",
      refused["config hash"],
      "otherwise a curve's two halves are different experiments")
for path in (wrong_fold, wrong_hash):
    path.unlink(missing_ok=True)
check("this run's own checkpoint passes both guards",
      state["fold"] == trainer.fold and state["config_hash"] == trainer.hash,
      f"fold {state['fold']}, hash {state['config_hash']}")

# -- the model learned something -----------------------------------------
final = history[-1]
held = final["metrics"]["per_dataset"].get(trainer.held_out)
check("validation Dice > 0 on the held-out dataset",
      bool(held) and held["dice"] > 0,
      f"{trainer.held_out}: dice {held['dice']:.4f}, IoU {held['iou']:.4f}, "
      f"boundary-F {held['boundary_f']:.4f}" if held else "held-out dataset "
      "absent from the validation split")
check("validation Dice > 0 pooled", final["metrics"]["pooled"]["dice"] > 0,
      f"pooled dice {final['metrics']['pooled']['dice']:.4f} (footnote)")

spread = (final["metrics"]["pooled"]["prob_max"]
          - final["metrics"]["pooled"]["prob_min"])
check("predictions are not all one value", spread > 0.01,
      f"probabilities span {final['metrics']['pooled']['prob_min']:.4f} .. "
      f"{final['metrics']['pooled']['prob_max']:.4f}, mean "
      f"{final['metrics']['pooled']['prob_mean']:.4f}")
check("the model predicts SOME boundary",
      final["metrics"]["pooled"]["pred_fraction"] > 0.001,
      f"pred_frac {final['metrics']['pooled']['pred_fraction']:.4f} vs "
      f"true_frac {final['metrics']['pooled']['true_fraction']:.4f} -- a "
      "pred_frac far below true_frac is the pos_weight failure")

# -- the unfreeze actually happened ---------------------------------------
freeze_epochs = int(model_settings["freeze_encoder_epochs"])
last_epoch = final["epoch"]
covered = freeze_epochs > 0 and last_epoch >= freeze_epochs
if covered:
    transition = trainer.freeze_transitions[0] if trainer.freeze_transitions else None
    check("trainable parameter count changed at the unfreeze epoch",
          bool(transition)
          and transition["trainable_after"] > transition["trainable_before"]
          and transition["optimized_after"] > transition["optimized_before"],
          f"epoch {transition['epoch']}: trainable "
          f"{transition['trainable_before']:,} -> {transition['trainable_after']:,}, "
          f"optimizer {transition['optimized_before']:,} -> "
          f"{transition['optimized_after']:,}" if transition else
          "no transition was recorded")
else:
    check("trainable parameter count changed at the unfreeze epoch", False,
          f"this run reached epoch {last_epoch} and the unfreeze epoch is "
          f"{freeze_epochs}; the assertion could not be made, which is not a pass")
check("the encoder is trainable at the end of the run",
      not final["encoder_frozen"],
      f"trainable {final['trainable_params']:,} params, optimizer holds "
      f"{final['optimized_params']:,}")

# -- per-dataset reporting is complete ------------------------------------
expected_datasets = set(summary["val_composition"])
reported = set(final["metrics"]["per_dataset"])
check("per-dataset metrics present for every dataset in val",
      reported == expected_datasets,
      f"reported {sorted(reported)}, val contains {sorted(expected_datasets)}")
check("every metric is present for every dataset",
      all(set(train_mod.METRIC_ORDER) <= set(m)
          for m in final["metrics"]["per_dataset"].values()),
      f"{list(train_mod.METRIC_ORDER)}")
check("pixel accuracy is not reported anywhere",
      not any("accuracy" in k for m in final["metrics"]["per_dataset"].values()
              for k in m),
      "boundaries are 5-15% of pixels; accuracy would read 85-95% for a model "
      "that predicts nothing")

# -- the clDice finding is recorded, not silently acted on ----------------
degenerate_epochs = sum(1 for r in history if r["cldice_probe"]["degenerate"])
check("the clDice diagnostic ran every epoch",
      all("cldice_probe" in r and r["cldice_probe"]["tiles"] > 0 for r in history),
      f"{degenerate_epochs} of {len(history)} epochs degenerate; final verdict: "
      f"{final['cldice_probe']['verdict']}")
check("loss weights were NOT changed in response to the diagnostic",
      float(loss_settings["w_cldice"]) == float(
          losses_mod.load_config()["w_cldice"]),
      f"w_cldice is still {loss_settings['w_cldice']} from configs/default.yaml; "
      "the finding is recorded in the report instead")

failed = [n for n, ok in checks if not ok]
print(f"\n{len(checks) - len(failed)}/{len(checks)} checks passed")
if failed:
    raise AssertionError("failed checks: " + ", ".join(failed))

## Save the checkpoint where it survives, write the report, push

The checkpoints were already written under `PERSISTENT_DIR` after every epoch —
that is the point of the atomic per-epoch save, not something done at the end.
This cell confirms where they landed, prints the path and size, and writes
`reports/train_<fold>.{json,md}`.

Only `reports/` and `configs/` are pushed. **Checkpoints are never committed**:
they are hundreds of megabytes of binary and the repo is not a model registry.
Step 7 reads `best.pt` from `PERSISTENT_DIR`, and the report names the exact
path and config hash it should be looking for.

In [ ]:
from scripts.push_results import push_results

for label, path in (("last", trainer.last_path), ("best", trainer.best_path)):
    if path.is_file():
        print(f"{label}: {path}  ({path.stat().st_size / 1024 ** 2:.0f} MB)")
    else:
        print(f"{label}: MISSING at {path}")

if PATHS["platform"] == "kaggle":
    print("\n!!! KAGGLE: /kaggle/working DOES NOT SURVIVE THIS SESSION.")
    print("    The checkpoints above are already effectively lost unless you")
    print("    save this notebook's output as a dataset version now. Treat a")
    print("    checkpoint left only in /kaggle/working as gone.")
else:
    print(f"\n{PATHS['platform']}: PERSISTENT_DIR is {PATHS['persistent_dir']}, "
          "which survives a session restart.")

md_path, json_path = train_mod.write_report(trainer)
print(f"\nwrote {md_path}")
print(f"wrote {json_path}")
print(f"\nstep 7 should load: {trainer.best_path}")
print(f"  fold {trainer.fold}, config hash {trainer.hash}, epoch "
      f"{trainer.best['epoch']}, {trainer.best['key']} Dice "
      f"{trainer.best['metric']:.4f}")

push_results(
    f"step 6: training run on {trainer.fold}",
    paths=PATHS,
    expect=[md_path, json_path],
)